# STASH — length scale & energy spectrum (for a future Tutorial 2)

> **Not wired into the site.** These cells were moved out of the Assignment 1
> material on 2026-09-09 to be reworked later into a separate tutorial on
> two-point correlations, the integral length scale, and the energy spectrum.
> Content is **unpolished** — original template cells, kept verbatim.


# Two point correlation and integral length scale


## Reordering the Data





In [ ]:
## Task 2.2

## Two point correlation along a single line


$$
R_{i, j}(\mathbf{r}) \equiv\left\langle u_i^{\prime}(\mathbf{x}+\mathbf{r}) u_j^{\prime}(\mathbf{x})\right\rangle
$$

In [ ]:
### Task 2.3

##two point correlation along single line
import numpy as np
import matplotlib.pyplot as plt

# here you can select the y coordinate (0-200) for the line parellel to the x axis that is used for the calculation
y_coordinate_line = 30

# Iterate over time steps
timestep_keys = sorted(timesteps_DNS_sliced.keys())  # necessary for iteration

for ts in timestep_keys[::1]:  # Filter timesteps
    # Extract columns
    x_coords = timesteps_DNS_sliced[ts]['x']
    y_coords = timesteps_DNS_sliced[ts]['y']
    vx = timesteps_DNS_sliced[ts]['velocity_x']

    # Create unique grid points and choose y for line
    x_unique = np.unique(x_coords)  # Unique x-values
    y_unique = np.unique(y_coords)  # Unique y-values
    y_line = y_unique[y_coordinate_line]  # Select the y-value for the line

    # Extract the components of velocity_x along the line
    mask = (x_coords.isin(x_unique)) & (y_coords == y_line)  # Mask for the line
    velocity_x_line = vx[mask]  # Extract velocity_x values along the line
    
    #Initialize arrays
    n_points = len(x_unique)  # Number of unique points
    correlation = np.zeros(n_points)  # To store the correlation
    counts = np.zeros(n_points)  # To store the counts for normalization
    velocity_x_line = velocity_x_line.values # ensure correct indices

    # Compute the two-point correlation using indices
    for i in range(n_points):
        for j in range(n_points):
            ## complete the code inside of the loop!
            # r_index =
            # correlations[] =
            # counts[] =
            

# Normalize the correlation
normalized_correlation = np.divide(
    correlation, counts, out=np.zeros_like(correlation), where=counts > 0
)

# Normalize the correlation by the autocorrelation (the value at r=0)
autocorrelation = normalized_correlation[0]
normalized_correlation /= autocorrelation  # Normalize all values by the autocorrelation

# Plot the two-point correlation
plt.xlabel('dimensionless length [-]')
plt.ylabel('correlation normalized over RST [-]')
plt.title('Two point correlation of velocity fluctuations along single line')
plt.plot(x_unique, normalized_correlation, marker='o', label="Two-Point Correlation")
plt.show()

## Two point correlation


In [ ]:
### Task 2.4

import numpy as np
import matplotlib.pyplot as plt

## builds the radial bins
def initialize_bins(X, Y, num_bins):
    max_distance = np.sqrt((X.max() - X.min())**2 + (Y.max() - Y.min())**2)
    r_bins = np.linspace(0, max_distance, num=num_bins)
    r_bin_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
    return r_bins, r_bin_centers

## constructs a 2D grid of velocity vectors from data
def construct_velocity_field(timestep_data):
    x_coords = timestep_data['x']
    y_coords = timestep_data['y']
    
    x_unique = np.unique(x_coords)
    y_unique = np.unique(y_coords)
    X, Y = np.meshgrid(x_unique, y_unique)
    
    vx = timestep_data['velocity_x']
    vy = timestep_data['velocity_y']   

    velocity_field = np.empty(X.shape, dtype=object)
    for row in timestep_data.itertuples(index=False):
        timestep, x, y, z, vx, vy, vz = row
        i = np.where(x_unique == x)[0][0]
        j = np.where(y_unique == y)[0][0]
        velocity_field[i, j] = (vx, vy)
        
    return velocity_field, X, Y

## calculates the two point correlation
def compute_radial_correlation(velocity_field, X, Y, r_bins, r_bin_centers, n_samples=4):   
    grid_shape = velocity_field.shape
    sample_indices = np.linspace(0, grid_shape[1] - 1, n_samples, dtype=int)
    sample_points = np.array([(i, j) for i in sample_indices for j in sample_indices])
    
    radial_correlation = np.zeros(len(r_bin_centers))
    counts = np.zeros(len(r_bin_centers))
    
    for i, j in sample_points:
        vx1, vy1 = velocity_field[i, j]
        x1, y1 = X[i, j], Y[i, j]
        
        for m in range(grid_shape[0]):
            for n in range(grid_shape[1]):
                vx2, vy2 = velocity_field[m, n]
                x2, y2 = X[m, n], Y[m, n]
                
                r = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)
                dot_product = vx1 * vx2 + vy1 * vy2
                
                for k in range(len(r_bin_centers)):
                    if r_bins[k] <= r < r_bins[k + 1]:
                        radial_correlation[k] += dot_product
                        counts[k] += 1
                        break
                        
    radial_correlation = np.divide(radial_correlation, counts, out=np.zeros_like(radial_correlation), where=counts > 0)
    return radial_correlation, counts


## plots the two point correlation
def plot_correlation(r_bin_centers, correlation):
    plt.xlabel('dimensionless length [-]')
    plt.ylabel('correlation normalized over RST [-]')
    plt.title('Two point correlation of velocity fluctuations on 2D plane')
    plt.plot(r_bin_centers, correlation, marker='o')
    plt.show()


## main function, iterates through time steps and calls other functions
def process_timesteps(timesteps_DNS_sliced, num_bins=100, sample_points=2):     # number of bins and sample points is set here
    global final_correlation, r_bin_centers #declare global so these can be used in further tasks

    timestep_keys = sorted(timesteps_DNS_sliced.keys())
    r_bins = r_bin_centers = None
    accumulated_correlation = accumulated_counts = None

    for ts in timestep_keys[::20]:      # set how to iterate through time, ::20 means that only every 20th time step is used
        velocity_field, X, Y = construct_velocity_field(timesteps_DNS_sliced[ts])
        
        if r_bins is None:
            r_bins, r_bin_centers = initialize_bins(X, Y, num_bins)
            accumulated_correlation = np.zeros(len(r_bin_centers))
            accumulated_counts = np.zeros(len(r_bin_centers))
        
        radial_correlation, counts = compute_radial_correlation(
            velocity_field, X, Y, r_bins, r_bin_centers, n_samples=sample_points
        )
        
        accumulated_correlation += radial_correlation
        accumulated_counts += counts

    final_correlation = np.divide(
        accumulated_correlation, accumulated_counts, out=np.zeros_like(accumulated_correlation), where=accumulated_counts > 0
    )

    final_correlation /= final_correlation[0]  # Normalize

    plot_correlation(r_bin_centers, final_correlation)


## call main function
process_timesteps(timesteps_DNS_sliced)


## Calculating a length scale




In [ ]:
### Task 2.5


### What does this length scale mean




In [ ]:
### Task 2.6

## calculation of length scale for a 0.7m^3 domain

# Energy spectrum and turbulent kinetic energy


## Calculating the turbulent energy spectrum 

$$\hat{E}(\boldsymbol{\kappa}, t) = \frac{1}{2} \langle \hat{u}_i^*(\boldsymbol{\kappa}, t) \hat{u}_i(\boldsymbol{\kappa}, t) \rangle$$


In [ ]:
## Task 2.7

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_parquet('isotropic_cube.parquet')
 
dim = 200
L = 2* np.pi * (200/1024)
dx = 2* np.pi *(1/1024)


u_array = np.zeros((dim, dim, dim), dtype=np.float32)
u_array[(df['x'] / dx).round().astype(int),
        (df['y'] / dx).round().astype(int),
        (df['z'] / dx).round().astype(int)] = df['velocity_x'].values

v_array = np.zeros((dim, dim, dim), dtype=np.float32)
v_array[(df['x'] / dx).round().astype(int),
        (df['y'] / dx).round().astype(int),
        (df['z'] / dx).round().astype(int)] = df['velocity_y'].values

w_array = np.zeros((dim, dim, dim), dtype=np.float32)
w_array[(df['x'] / dx).round().astype(int),
        (df['y'] / dx).round().astype(int),
        (df['z'] / dx).round().astype(int)] = df['velocity_z'].values


uu_fft=np.fft.fftn(u_array)
vv_fft=np.fft.fftn(v_array)
ww_fft=np.fft.fftn(w_array)

uu_fft=(np.abs(uu_fft)/dim**3)**2
vv_fft=(np.abs(vv_fft)/dim**3)**2
ww_fft=(np.abs(ww_fft)/dim**3)**2



dk = 2 * np.pi / L
k_end=int(dim/2)
k = (np.arange(k_end) + 1) * dk
rx=np.array(range(dim))-dim/2+1
rx=np.roll(rx,int(dim/2)+1)


r=np.zeros((rx.shape[0],rx.shape[0],rx.shape[0]))
for i in range(rx.shape[0]):
    for j in range(rx.shape[0]):
            r[i,j,:]=rx[i]**2+rx[j]**2+rx[:]**2
r=np.sqrt(r)

k=(np.array(range(k_end))+1)*dk

bins=np.zeros((k.shape[0]+1))
for N in range(k_end):
    if N==0:
        bins[N]=0
    else:
        bins[N]=(k[N]+k[N-1])/2    
bins[-1]=k[-1]

inds = np.digitize(r*dk, bins, right=True)
spectrum=np.zeros((k.shape[0]))
bin_counter=np.zeros((k.shape[0]))

for N in range(k_end):
    spectrum[N]=(np.sum(uu_fft[inds==N+1])+np.sum(vv_fft[inds==N+1])+np.sum(ww_fft[inds==N+1]))
    bin_counter[N]=np.count_nonzero(inds==N+1)

spectrum=spectrum*2*np.pi*(k**2)/(bin_counter*dk**3)


plt.xlabel(r'$\kappa$')
plt.ylabel(r'$E(\kappa)$')
plt.title('Energy spectrum')
plt.loglog(k, spectrum)
plt.show()